In [1]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime
print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model, Verbose
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

added '/home/holo/dev/PFE_LLM_art_generation' to import paths
sys.version_info(major=3, minor=10, micro=19, releaselevel='final', serial=0)


In [2]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [3]:
import importlib
import LLM.model
_ = importlib.reload(LLM.model)
from LLM.model import Model

In [6]:
try:
    torch.cuda.empty_cache()
    del model # type: ignore
    torch.cuda.empty_cache()
except Exception: pass
model = Model(
    save_name="tmp-tests", tokenizer=TOKENIZER_SAVE_DIRECTORY.joinpath("our_tokenizer"), 
    device="cuda", depth=6, head_dim=128, context_size=4096, nb_heads_mult=1.0)
model.show_infos()

loading the tokenizer from: /home/holo/dev/PFE_LLM_art_generation/tokenizer_save/our_tokenizer.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(128/768) = 2.449490
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=1, n_kv_head=1, n_embd=128, window_pattern='SSSL')
1_589_356 total params (embeding: 327_680 | last layer: 81_920 | transformer: 1_179_744)
on device: device(type='cuda', index=0), with effective context_size: 4096


In [ ]:
ID_1, _ = model.save("un_trained")

saving the tokenizer to: /home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/tokenizer.json
saving the historique to: /home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/versions/version_0001_un_trained/history.json


(1,
 PosixPath('/home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/versions/version_0001_un_trained'))

In [7]:
try:
    if "dataset" in globals() : raise FileExistsError
    dataset = svg_dataset.SVGDataset(
        OUR_DATASET_DIRECTORY, context_size=model.context_size,
        tokenizer=model.tokenizer.encode, decoder=model.tokenizer.decode)
except FileExistsError: pass

In [11]:
model.train(
    dataset=dataset, batch_size=16, 
    nbEpoches=1, timeLimite=999_999, verbose=Verbose.liveProgress)


starting epoch: 1
finished batches in 1 min 56.8 sec              
finished epoches in 1 min 56.8 sec
trained on: 863 batch (13793 chuncks) in 1 min 46.1 sec
 -> 8.13 batch/sec | 130.02 chuncks/sec
 -> CE: 1.022 | PPL: 2.778 | top-1: 74.90%


In [ ]:
ID_2, _ = model.save("trained-1epoches")

saving the tokenizer to: /home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/tokenizer.json
saving the historique to: /home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/versions/version_0002_trained-1epoches/history.json


(2,
 PosixPath('/home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/versions/version_0002_trained-1epoches'))

In [ ]:
new_model = Model.load("tmp-tests", ID_2, torch.device("cuda:0"))

loading the tokenizer from: /home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/tokenizer.json
loading the historique from: /home/holo/dev/PFE_LLM_art_generation/models_save/tmp-tests/versions/version_0002_trained-1epoches/history.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(128/768) = 2.449490


In [14]:
new_model.train(
    dataset=dataset, batch_size=16, 
    nbEpoches=1, timeLimite=999_999, verbose=Verbose.liveProgress)


starting epoch: 1
finished batches in 2 min 1.6 sec               
finished epoches in 2 min 1.7 sec
trained on: 863 batch (13793 chuncks) in 1 min 47.3 sec
 -> 8.04 batch/sec | 128.53 chuncks/sec
 -> CE: 0.7856 | PPL: 2.194 | top-1: 80.72%
